# Genixcure PacBio HiFi — EcoRV-linear plasmid reference alignment (v1)

**실험 조건:** 제공받은 SnapGene `.dna`는 **6,848 bp 원형 plasmid**, EcoRV 절단 부위 **1개**, 실제 PacBio 투입은 **EcoRV-digested linear DNA**. 따라서 `Flye` 원형 조립이 아니라 **기존 확정 plasmid 참조서열에 선형 형태로 직접 정렬**합니다.

- EcoRV 인식: `GAT|ATC` — 원본 1-based **4,716|4,717** 절단. 선형 서열 시작 = 원본 4,717번 염기, 마지막 = 4,716번 염기.
- 소스 파일의 `W` 3개(원본 51–53)는 모호한 IUPAC 염기로, 원본 FASTA에 보존합니다. minimap2 정렬용에는 해당 위치를 `N`으로 표시합니다. 임의로 특정 A/T 대립염기를 채우지 않습니다.
- `UMI 12mer`, `Barcode 12mer`, `6bpUID`와 같은 주석이 발견되었습니다. **실제로 Read 내 가변 barcode/UMI가 몇 염기인지 확인 전에는 이를 제거하거나 biological variant라고 확정하지 않습니다.**
- 이전 `PLA_6_13_hifi_unique_sequences.csv.gz` (`sequence,read_count,length_bp`)에서 확률적 가중 unique sample을 추출해 진단하고, **플라스미드 원형으로 합치지 않습니다.**
- 이 분석은 **비파괴적 파일럿**입니다. 122.7GB BAM을 다시 다운로드하지 않으며, 원래 CSV도 덮어쓰지 않습니다.

**주의:** EcoRV single-cut reference는 실험 설계의 기대값입니다. 실제 완전 절단율, chimeric read, truncated fragment, ligation/concatemer 및 sample contamination은 PacBio alignment로 검증해야 합니다.


In [ ]:
import subprocess, shutil
if shutil.which('minimap2') is None:
    subprocess.run(['apt-get','update','-qq'], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(['apt-get','install','-y','-qq','minimap2'], check=True, stdout=subprocess.DEVNULL)
try:
    from Bio import SeqIO
    from Bio.Restriction import EcoRV
except ImportError:
    subprocess.run(['python','-m','pip','-q','install','biopython'],check=True)
    from Bio import SeqIO
    from Bio.Restriction import EcoRV
subprocess.run(['minimap2','--version'],check=True)
print('Reference alignment tools installed; Flye is NOT required.')


In [ ]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import csv, gzip, hashlib, heapq, json, math, random, time, shutil, os

drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13')
REF_DIR=BASE/'Reference_EcoRV_Linear'
SOURCE_DNA=REF_DIR/'Genixcure_plasmid_260915.dna'
UNIQUE_CSV=BASE/'PLA_6_13_hifi_unique_sequences.csv.gz'
OUT=REF_DIR/'Alignment_Pilot_v1'
WORK=Path('/content/genixcure_pacbio_ecorv_linear')
OUT.mkdir(parents=True,exist_ok=True); WORK.mkdir(parents=True,exist_ok=True)

PILOT_UNIQUE=12000     # 대표 weighted unique reads 수
PILOT_SEED=20261104
MIN_IDENTITY=0.985    # 파일럿 진단 임계값, 변이 판정 기준이 아님
MIN_QUERY_COV=0.90
MIN_REFERENCE_COV=0.90
TERMINAL_TOL_BP=150
REUSE_EXISTING_PILOT=True
RUN_PILOT_ALIGNMENT=True

assert SOURCE_DNA.is_file(), f'SnapGene file missing: {SOURCE_DNA}'
print('SnapGene source:',SOURCE_DNA)
print('PacBio unique CSV present:',UNIQUE_CSV.exists())


## 1. EcoRV single-cut linear reference 제작 및 무결성 검증

코드는 실제 SnapGene 바이너리에서 sequence와 topology를 읽습니다. `EcoRV.search(...,linear=False)`가 1개가 아니면 **중단**합니다. 원본 IUPAC 서열은 별도로 보존하며, 정렬용은 모호한 염기 3개만 N으로 바꿉니다.


In [ ]:
import argparse
import collections
import hashlib
import json
from pathlib import Path

from Bio import SeqIO
from Bio.Restriction import EcoRV


def digest(record):
    seq = str(record.seq).upper()
    if record.annotations.get('topology', '').lower() != 'circular':
        raise ValueError('SnapGene record topology is not circular; cannot infer a single EcoRV linearization')
    sites = list(EcoRV.search(record.seq, linear=False))
    if len(sites) != 1:
        raise ValueError(f'Expected exactly ONE EcoRV cleavage site: found {len(sites)} at {sites}')
    cut_1based = int(sites[0])  # 1-based base *after* cut
    cut_0based = cut_1based - 1
    motif = (seq + seq)[cut_0based-3:cut_0based+3]
    if motif != 'GATATC':
        raise AssertionError(f'EcoRV site mismatch at expected cut {cut_1based}: {motif}')
    linear = seq[cut_0based:] + seq[:cut_0based]
    assert len(linear) == len(seq)
    assert linear[-3:] + linear[:3] == 'GATATC'
    annotations = []
    for feat in record.features:
        labels = feat.qualifiers.get('label') or []
        label = labels[0] if isinstance(labels, list) else str(labels)
        if label:
            annotations.append({'label': label, 'type': feat.type,
                                'start_1based': int(feat.location.start) + 1,
                                'end_1based_inclusive': int(feat.location.end),
                                'strand': feat.location.strand})
    return linear, {
        'source_topology': record.annotations.get('topology'),
        'plasmid_length_bp': len(seq),
        'ecorv_site_count': len(sites),
        'ecorv_recognition': 'GATATC',
        'ecorv_recognition_first_base_1based': cut_1based-3,
        'ecorv_recognition_last_base_1based': cut_1based+2,
        'ecorv_cut_between_original_bases_1based': [cut_1based-1, cut_1based],
        'linear_reference_original_first_base_1based': cut_1based,
        'linear_reference_original_last_base_1based': cut_1based-1,
        'iupac_ambiguous_bases': dict(collections.Counter(x for x in seq if x not in 'ACGT')),
        'iupac_ambiguous_positions_original_1based': [i+1 for i,ch in enumerate(seq) if ch not in 'ACGT'],
        'feature_annotations': annotations,
        'important_warning': 'UMI/Barcode feature LABELS are present but do not establish actual read-variable region lengths; do not remove or group these before alignment.'
    }


def write_fasta(path, name, seq):
    with Path(path).open('w') as out:
        out.write(f'>{name}\n')
        for i in range(0, len(seq), 80):
            out.write(seq[i:i+80] + '\n')


def create(source, output):
    source = Path(source)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    record = SeqIO.read(str(source), 'snapgene')
    seq = str(record.seq).upper()
    linear, meta = digest(record)
    meta['snapgene_source_filename'] = source.name
    meta['snapgene_source_sha256'] = hashlib.sha256(source.read_bytes()).hexdigest()
    meta['reference_linear_sha256'] = hashlib.sha256(linear.encode()).hexdigest()
    meta['mapped_reference_ambiguous_to_N'] = True
    write_fasta(output/'plasmid_original_circular_IUPAC.fasta', 'Genixcure_original_circular_6848bp', seq)
    write_fasta(output/'plasmid_EcoRV_linear_IUPAC.fasta', 'Genixcure_EcoRV_linear_6848bp_IUPAC', linear)
    write_fasta(output/'plasmid_EcoRV_linear_minimap2.fasta', 'Genixcure_EcoRV_linear_6848bp',
                ''.join(b if b in 'ACGT' else 'N' for b in linear))
    (output/'EcoRV_reference_manifest.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    return meta


def self_test():
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord
    # One EcoRV site, distinct flanks, reversible rotation, and blunt-end junction.
    record = SeqRecord(Seq('AACCGATATCTTGGCCAA'))
    record.annotations['topology'] = 'circular'
    linear, m = digest(record)
    assert len(linear) == 18
    assert m['ecorv_site_count'] == 1
    assert m['ecorv_cut_between_original_bases_1based'] == [7, 8]
    assert linear == 'ATCTTGGCCAAAACCGAT'
    assert linear[-3:] + linear[:3] == 'GATATC'
    print('EcoRV circular→linear reference self-test PASS')


META=create(SOURCE_DNA,REF_DIR)
LINEAR_LEN=META['plasmid_length_bp']
REF_FASTA=REF_DIR/'plasmid_EcoRV_linear_minimap2.fasta'
assert LINEAR_LEN==6848 and META['ecorv_cut_between_original_bases_1based']==[4716,4717]
print('Linear reference length:',LINEAR_LEN,'bp; EcoRV 1-based cut:',META['ecorv_cut_between_original_bases_1based'])
print('Reference:',REF_FASTA)


## 2. 대표 HiFi unique read 준비

이전 원형 조립 파일럿과 **동일한 런타임**이라면 `/content/plasmid_circular_pilot/weighted_pilot_reads.fasta`와 `sample_metadata.csv`를 재사용합니다. 그렇지 않으면 9.65GB gz CSV를 1회 순차 스캔해 `read_count` 기반 가중 reservoir sampling을 수행합니다. **전체 unique 서열의 개수/분포가 표본 12,000개로 대표된다고 확정할 수 없으며, 가중 표본은 읽기 빈도에 치우쳐집니다.**


In [ ]:
PILOT_FASTA=WORK/'weighted_pilot_reads.fasta'
PILOT_META=WORK/'sample_metadata.csv'
OLD_WORK=Path('/content/plasmid_circular_pilot')
if REUSE_EXISTING_PILOT and (OLD_WORK/'weighted_pilot_reads.fasta').is_file() and (OLD_WORK/'sample_metadata.csv').is_file():
    shutil.copyfile(OLD_WORK/'weighted_pilot_reads.fasta',PILOT_FASTA)
    shutil.copyfile(OLD_WORK/'sample_metadata.csv',PILOT_META)
    print('Reused previous weighted pilot FASTA and metadata',flush=True)
elif REUSE_EXISTING_PILOT and (OUT/'weighted_pilot_reads.fasta').is_file() and (OUT/'sample_metadata.csv').is_file():
    shutil.copyfile(OUT/'weighted_pilot_reads.fasta',PILOT_FASTA)
    shutil.copyfile(OUT/'sample_metadata.csv',PILOT_META)
    print('Reused saved pilot FASTA and metadata from Drive',flush=True)
else:
    assert UNIQUE_CSV.is_file(), f'PacBio unique CSV not found: {UNIQUE_CSV}'
    rng=random.Random(PILOT_SEED)
    heap=[]; reads=0; n_unique=0; singletons=0; started=time.monotonic()
    with gzip.open(UNIQUE_CSV,'rt',newline='') as f:
        reader=csv.DictReader(f)
        assert reader.fieldnames==['sequence','read_count','length_bp'],reader.fieldnames
        for row in reader:
            seq=row['sequence'].upper(); ct=int(row['read_count']); L=int(row['length_bp'])
            if ct<1 or len(seq)!=L or not seq or set(seq)-set('ACGTN'):
                raise ValueError(f'Invalid unique sequence row {n_unique+1}')
            n_unique+=1; reads+=ct; singletons+=(ct==1)
            score=-math.log(max(rng.random(),1e-300))/ct
            item=(-score,n_unique,seq,ct)
            if len(heap)<PILOT_UNIQUE:heapq.heappush(heap,item)
            elif item>heap[0]:heapq.heapreplace(heap,item)
            if n_unique%250000==0:
                print(f'Unique rows scanned={n_unique:,}; elapsed={(time.monotonic()-started)/60:.1f} min',flush=True)
    print(f'CSV check: unique={n_unique:,}, reads={reads:,}, singletons={singletons:,}',flush=True)
    with PILOT_FASTA.open('w') as fa,PILOT_META.open('w',newline='') as meta:
        writer=csv.writer(meta);writer.writerow(['query_id','read_count','length_bp'])
        for i,(_,_,seq,ct) in enumerate(sorted(heap,reverse=True),1):
            qid=f'pilot_{i:06d}'
            fa.write(f'>{qid}\n{seq}\n')
            writer.writerow([qid,ct,len(seq)])
    shutil.copyfile(PILOT_FASTA,OUT/'weighted_pilot_reads.fasta')
    shutil.copyfile(PILOT_META,OUT/'sample_metadata.csv')
assert PILOT_FASTA.is_file() and PILOT_META.is_file()
assert PILOT_FASTA.stat().st_size>100 and PILOT_META.stat().st_size>100
print('Pilot FASTA:',PILOT_FASTA.stat().st_size,'bytes')


## 3. Minimap2 HiFi mapping: linear reference only

`-x map-hifi`, `--secondary=no`: 한 read당 primary match를 파일럿에서 대표값으로 선택합니다. 사용된 reference는 **6,848bp 단일 선형 분자**이며, 과거의 `reference+reference` 원형 매핑은 수행하지 않습니다. 정렬 상태를 완전 길이, 부분 산물, 긴 다중체 후보, 저일치/비매핑으로 분리합니다.


In [ ]:
if RUN_PILOT_ALIGNMENT:
    PILOT_PAF=WORK/'pilot_EcoRV_linear.paf'
    cmd=['minimap2','-x','map-hifi','--secondary=no','-c','-t','4',str(REF_FASTA),str(PILOT_FASTA)]
    print('Running:', ' '.join(cmd),flush=True)
    with PILOT_PAF.open('w') as output,(WORK/'minimap2_linear.log').open('w') as err:
        run=subprocess.run(cmd,stdout=output,stderr=err)
    if run.returncode:
        print((WORK/'minimap2_linear.log').read_text(errors='replace')[-8000:])
        raise RuntimeError(f'minimap2 failed with exit={run.returncode}')
    print('Mapped PAF bytes:',PILOT_PAF.stat().st_size)


## 4. Full-length 분포 및 정렬 QC

`minimap2` PAF query/target coverage, identity, strand, read length를 기록합니다. 아래 기준은 **탐색적 분류**이지 분자종이나 단백질/임상 변이 확정 기준이 아닙니다. 여러 segment의 split alignment는 본 파일럿의 단일 primary PAF만으로 배제할 수 없습니다.


In [ ]:
meta={}
with PILOT_META.open(newline='') as f:
    for r in csv.DictReader(f):
        meta[r['query_id']]={'read_count':int(r['read_count']),'length_bp':int(r['length_bp'])}

def read_paf(path):
    best={}
    with path.open() as f:
        for line in f:
            fields=line.rstrip('\n').split('\t')
            if len(fields)<12:continue
            q=fields[0];ql=int(fields[1]);qs=int(fields[2]);qe=int(fields[3]);strand=fields[4]
            tlen=int(fields[6]);ts=int(fields[7]);te=int(fields[8]);match=int(fields[9]);alen=int(fields[10]);mq=int(fields[11])
            if q not in meta or ql<1 or alen<1:continue
            is_secondary=any(tag=='tp:A:S' for tag in fields[12:])
            if is_secondary:continue
            cand={'q_length':ql,'q_start':qs,'q_end':qe,'strand':strand,'ref_start':ts,'ref_end':te,
                  'qcov':round((qe-qs)/ql,6),'rcov':round((te-ts)/tlen,6),
                  'identity':round(match/alen,6),'mapq':mq,'target_length':tlen}
            rank=(match,qe-qs,mq)
            if q not in best or rank>best[q][0]:best[q]=(rank,cand)
    return {k:v[1] for k,v in best.items()}

def classify_alignment(m):
    if m is None:return 'unmapped'
    if m['q_length']>1.5*LINEAR_LEN:return 'long_or_concatemer_candidate'
    if m['qcov']<MIN_QUERY_COV:return 'split_or_partial_query_candidate'
    if m['identity']<MIN_IDENTITY:return 'divergent_or_error_candidate'
    if m['rcov']>=MIN_REFERENCE_COV and 0.80*LINEAR_LEN<=m['q_length']<=1.25*LINEAR_LEN:
        if m['ref_start']<=TERMINAL_TOL_BP and (LINEAR_LEN-m['ref_end'])<=TERMINAL_TOL_BP:
            return 'near_full_length_linear_plasmid'
        return 'near_full_length_but_terminals_incomplete'
    if m['rcov']>=0.50:return 'partial_plasmid_fragment'
    return 'short_plasmid_fragment'

assert classify_alignment(None)=='unmapped'
assert classify_alignment({'q_length':6848,'qcov':0.99,'identity':0.998,'rcov':0.99,'ref_start':15,'ref_end':6820})=='near_full_length_linear_plasmid'
assert classify_alignment({'q_length':11000,'qcov':0.99,'identity':0.998,'rcov':0.99,'ref_start':0,'ref_end':6848})=='long_or_concatemer_candidate'

alignments=read_paf(PILOT_PAF)
counts=Counter(); weighted=Counter(); lengths_by_category=Counter()
report=OUT/'pilot_EcoRV_linear_alignment_per_unique.csv.gz'
fields=['query_id','read_count','read_length','classification','qcov','rcov','identity','strand','mapq','ref_start','ref_end']
with gzip.open(report,'wt',newline='') as f:
    wr=csv.DictWriter(f,fieldnames=fields);wr.writeheader()
    for q, info in meta.items():
        al=alignments.get(q);cat=classify_alignment(al)
        counts[cat]+=1;weighted[cat]+=info['read_count']
        obj={'query_id':q,'read_count':info['read_count'],'read_length':info['length_bp'],
             'classification':cat}
        for k in fields[4:]:obj[k]=al.get(k) if al else ''
        wr.writerow(obj)
assert sum(counts.values())==len(meta)
assert sum(weighted.values())==sum(x['read_count'] for x in meta.values())
summary={'reference_length':LINEAR_LEN,'reference_file':str(REF_FASTA),
         'ecorv_cut_1based':META['ecorv_cut_between_original_bases_1based'],
         'pilot_unique':len(meta),'pilot_weighted_reads':sum(x['read_count'] for x in meta.values()),
         'pilot_classification_unique':dict(counts),'pilot_classification_weighted_reads':dict(weighted),
         'thresholds':{'min_identity':MIN_IDENTITY,'min_query_coverage':MIN_QUERY_COV,
                       'min_reference_coverage':MIN_REFERENCE_COV,'terminal_tolerance_bp':TERMINAL_TOL_BP},
         'important_note':'Representative weighted unique sample only; not population read fractions. UMI/barcode loci not yet validated.'}
(OUT/'pilot_EcoRV_linear_alignment_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print('Count conservation PASS; pilot unique:',len(meta))
for name,n in counts.most_common():
    print(f'{name:42s} unique={n:6,d} weighted_reads={weighted[name]:8,d}')
print('Output:',report)


## 5. 다음 실험 판단 기준

- **명확한 `near_full_length_linear_plasmid`**: 6,848bp 전체를 거의 덮고 양쪽 EcoRV 절단 말단 근처에 alignment 끝이 존재. Read 길이 분포 및 정확도와 함께 검증.
- **부분/짧은 Read**: 단편화, incomplete CCS, structural variant 혹은 sample preparation 문제. 무조건 불량·별개 plasmid로 판정하지 않음.
- **유사도가 낮거나 정렬되지 않는 Read**: 실제 서로 다른 construct, adapter, rearrangement, sequencing error 가능성을 구분하려면 추가 검증.
- **희귀 변이/Unique 과잉**: SnapGene `UMI 12mer`, `Barcode 12mer` **주석**을 발견했습니다. 실제 가변 영역은 **Read-to-reference CIGAR를 확인**해 추론해야 하며, annotation label만으로 UMI 12bp가 실제 도입됐다고 결론내리지 않습니다.
- 실제 PacBio의 검증된 insert alignment가 얻어지면, **reference별 coverage, SNV/indel, ITR 구간 및 EcoRV 절단말단, variant-specific consensus**를 수행합니다. 이 단계에서는 자동으로 변이체를 합치지 않습니다.

**결과 경로:** `MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13/Reference_EcoRV_Linear/Alignment_Pilot_v1`.
